# Solving the Steady-State 2D Radiative Transfer Equation (RTE) using Vanilla-PINNs

In this notebook, we solve the steady-state 2D Radiative Transfer Equation (RTE) in a participating square medium using a **VANILLA-PINN** approach. This case corresponds to **Case 3b** of Biswal et al. (JQSRT, 2025): a cold enclosure with an internal radiation source.

### Case 3b (2D)
We focus on a 2D space + 3D direction test case with the following configuration:
* **Geometry:** $0 \le x \le L_x$ and $0 \le y \le L_y$ with $L_x = L_y = 1.0\text{ m}$ (Square medium).
* **Directions:** Direction of propagation $\vec{s}$ defined by spherical angles $(\theta, \phi)$, with directional cosines:
  * $\mu = \sin\theta \cos\phi$ (x-axis)
  * $\eta = \sin\theta \sin\phi$ (y-axis)
* **Equation (2D RTE with internal source):**
  $$\mu \frac{\partial I}{\partial x} + \eta \frac{\partial I}{\partial y} + (\kappa + \sigma) I(x, y, \mu, \eta) = \frac{\sigma}{4\pi} G(x, y) + S(x, y)$$
  where $G(x, y) = \int_{0}^{2\pi} \int_{0}^{\pi} I(x, y, \theta', \phi') \sin\theta' d\theta' d\phi'$.
* **Internal source:** $S(x, y) = 0.5 - r_0$ if $r_0 \le 0.5$, else $0$, with $r_0 = \sqrt{(x - 0.5)^2 + (y - 0.9)^2}$ (centered at $(0.5, 0.9)$).
* **Physical Properties:**
  * Scattering coefficient: $\sigma = 1.0\text{ m}^{-1}$ (Isotropic scattering).
  * Absorption coefficient: $\kappa = 0.5\text{ m}^{-1}$ (Absorbing and scattering medium; paper range $0.1 \le \kappa \le 2$).
* **Boundary Conditions (BC) on incoming directions:**
  * All four walls cold (non-emissive): $I = 0$ for incoming directions.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. PINN Architecture Definition
Definition of the standard Neural Network (Multi-Layer Perceptron) to approximate the intensity $I(x, y, \mu, \eta)$.

In [ ]:
class PinnRFEEq2D(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(4, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 2. Double Quadrature Setup for the Unit Sphere
Setting up a tensor product of Gauss-Legendre quadratures for $\xi = \cos\theta \in [-1, 1]$ and $\phi \in [0, 2\pi]$ to perform solid angle integration.

In [ ]:
def init_quadrature(N_theta=8, N_phi=16, device='cpu'):
    nodes_xi, weights_xi = np.polynomial.legendre.leggauss(N_theta)
    nodes_xi = 0.5 * (nodes_xi + 1.0)

   
    nodes_phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    weights_phi = np.full(N_phi, 2.0 * np.pi / N_phi)

    quad_mu_list = []
    quad_eta_list = []
    quad_w_list = []

    for i in range(N_theta):
        for j in range(N_phi):
            xi = nodes_xi[i]
            phi = nodes_phi[j]
            w = weights_xi[i] * weights_phi[j]
            
            mu = np.sqrt(1.0 - xi**2) * np.cos(phi)
            eta = np.sqrt(1.0 - xi**2) * np.sin(phi)
            
            quad_mu_list.append(mu)
            quad_eta_list.append(eta)
            quad_w_list.append(w)

    quad_mu = torch.tensor(quad_mu_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_eta = torch.tensor(quad_eta_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_w = torch.tensor(quad_w_list, dtype=torch.float32).view(-1, 1).to(device)
    return quad_mu, quad_eta, quad_w

## 3. Sampling and Collocation Points Generation
Functions to generate collocation points in the 4D domain $(x, y, \mu, \eta)$ and boundary points on the four limits for incoming directions.

In [ ]:
def generer_points_collocation(n_pde):
    x = torch.rand(n_pde, 1)
    y = torch.rand(n_pde, 1)
    xi = torch.rand(n_pde, 1) * 2.0 - 1.0
    phi = torch.rand(n_pde, 1) * 2.0 * np.pi
    mu = torch.sqrt(1.0 - xi**2) * torch.cos(phi)
    eta = torch.sqrt(1.0 - xi**2) * torch.sin(phi)
    return x.float(), y.float(), mu.float(), eta.float()

def generer_points_bords(n_bords):
    n_edge = n_bords // 4
    
    # Left edge: x = 0, y in [0,1], mu > 0 (phi in [-pi/2, pi/2])
    x_left = torch.zeros(n_edge, 1)
    y_left = torch.rand(n_edge, 1)
    xi_left = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_left = (torch.rand(n_edge, 1) - 0.5) * np.pi
    mu_left = torch.sqrt(1.0 - xi_left**2) * torch.cos(phi_left)
    eta_left = torch.sqrt(1.0 - xi_left**2) * torch.sin(phi_left)
    
    # Right edge: x = 1, y in [0,1], mu < 0 (phi in [pi/2, 3*pi/2])
    x_right = torch.ones(n_edge, 1)
    y_right = torch.rand(n_edge, 1)
    xi_right = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_right = torch.rand(n_edge, 1) * np.pi + 0.5 * np.pi
    mu_right = torch.sqrt(1.0 - xi_right**2) * torch.cos(phi_right)
    eta_right = torch.sqrt(1.0 - xi_right**2) * torch.sin(phi_right)
    
    # Bottom edge: x in [0,1], y = 0, eta > 0 (phi in [0, pi])
    x_bottom = torch.rand(n_edge, 1)
    y_bottom = torch.zeros(n_edge, 1)
    xi_bottom = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_bottom = torch.rand(n_edge, 1) * np.pi
    mu_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.cos(phi_bottom)
    eta_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.sin(phi_bottom)
    
    # Top edge: x in [0,1], y = 1, eta < 0 (phi in [pi, 2*pi])
    x_top = torch.rand(n_edge, 1)
    y_top = torch.ones(n_edge, 1)
    xi_top = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_top = torch.rand(n_edge, 1) * np.pi + np.pi
    mu_top = torch.sqrt(1.0 - xi_top**2) * torch.cos(phi_top)
    eta_top = torch.sqrt(1.0 - xi_top**2) * torch.sin(phi_top)
    
    return (
        x_left.float(), y_left.float(), mu_left.float(), eta_left.float(),
        x_right.float(), y_right.float(), mu_right.float(), eta_right.float(),
        x_bottom.float(), y_bottom.float(), mu_bottom.float(), eta_bottom.float(),
        x_top.float(), y_top.float(), mu_top.float(), eta_top.float(),
    )

## 4. Loss Functions Definition
Loss calculation for boundary conditions (BC) and the PDE residual (CLP).

In [ ]:
CENTRE = (0.5, 0.9)

def source_S(x, y):
    r0 = torch.sqrt((x - CENTRE[0]) ** 2 + (y - CENTRE[1]) ** 2)
    return torch.clamp(0.5 - r0, min=0.0)

def source_S_np(x, y):
    r0 = np.sqrt((x - CENTRE[0]) ** 2 + (y - CENTRE[1]) ** 2)
    return np.maximum(0.5 - r0, 0.0)

def calc_bc_loss(model,
                 x_l, y_l, mu_l, eta_l,
                 x_r, y_r, mu_r, eta_r,
                 x_b, y_b, mu_b, eta_b,
                 x_t, y_t, mu_t, eta_t):

    pred_l = model(torch.cat([x_l, y_l, mu_l, eta_l], dim=1))
    pred_r = model(torch.cat([x_r, y_r, mu_r, eta_r], dim=1))
    pred_b = model(torch.cat([x_b, y_b, mu_b, eta_b], dim=1))
    pred_t = model(torch.cat([x_t, y_t, mu_t, eta_t], dim=1))

    loss_l = torch.mean((pred_l - 0.0) ** 2)
    loss_r = torch.mean((pred_r - 0.0) ** 2)
    loss_b = torch.mean((pred_b - 0.0) ** 2)
    loss_t = torch.mean((pred_t - 0.0) ** 2)

    return loss_l + loss_r + loss_b + loss_t

def calc_clp_loss(model, x, y, mu, eta, quad_mu, quad_eta, quad_w, kappa=0.5, sigma=1.0):
    x.requires_grad_(True)
    y.requires_grad_(True)

    I_pred = model(torch.cat([x, y, mu, eta], dim=1))

    I_x = torch.autograd.grad(
        outputs=I_pred,
        inputs=x,
        grad_outputs=torch.ones_like(I_pred),
        create_graph=True,
    )[0]

    I_y = torch.autograd.grad(
        outputs=I_pred,
        inputs=y,
        grad_outputs=torch.ones_like(I_pred),
        create_graph=True,
    )[0]

    N = x.shape[0]
    N_q = quad_mu.shape[0]

    x_expanded = x.repeat(1, N_q)
    y_expanded = y.repeat(1, N_q)
    mu_expanded = quad_mu.t().repeat(N, 1)
    eta_expanded = quad_eta.t().repeat(N, 1)

    inputs_quad = torch.stack([x_expanded, y_expanded, mu_expanded, eta_expanded], dim=2).view(-1, 4)
    I_quad_preds = model(inputs_quad).view(N, N_q)

    G = torch.sum(I_quad_preds * quad_w.t(), dim=1, keepdim=True)

    residual = mu * I_x + eta * I_y + (kappa + sigma) * I_pred - (sigma / (4.0 * np.pi)) * G - source_S(x, y)

    loss_pde = torch.mean(residual ** 2)
    return loss_pde

## 5. Hardware (Device), Model, and Optimizer Initialization
Hardware detection, model creation, training data generation and optimizer definition.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

quad_mu, quad_eta, quad_w = init_quadrature(N_theta=16, N_phi=32, device=device)

n_pde = 15000
n_bords = 4000
kappa = 0.5
sigma = 1.0

x_colloc, y_colloc, mu_colloc, eta_colloc = generer_points_collocation(n_pde)
x_colloc = x_colloc.to(device)
y_colloc = y_colloc.to(device)
mu_colloc = mu_colloc.to(device)
eta_colloc = eta_colloc.to(device)

(
    x_l, y_l, mu_l, eta_l,
    x_r, y_r, mu_r, eta_r,
    x_b, y_b, mu_b, eta_b,
    x_t, y_t, mu_t, eta_t
) = generer_points_bords(n_bords)

x_l, y_l, mu_l, eta_l = x_l.to(device), y_l.to(device), mu_l.to(device), eta_l.to(device)
x_r, y_r, mu_r, eta_r = x_r.to(device), y_r.to(device), mu_r.to(device), eta_r.to(device)
x_b, y_b, mu_b, eta_b = x_b.to(device), y_b.to(device), mu_b.to(device), eta_b.to(device)
x_t, y_t, mu_t, eta_t = x_t.to(device), y_t.to(device), mu_t.to(device), eta_t.to(device)

modele = PinnRFEEq2D(hidden_dim=64, num_layers=4).to(device)

## 6. Model Training (Adam)
Training phase of the PINN model using the Adam optimizer.

In [ ]:
optimizer = optim.Adam(modele.parameters(), lr=0.005)
epochs = 2000

for epoch in range(epochs):
    optimizer.zero_grad()
    loss_bc = calc_bc_loss(modele, 
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc, quad_mu, quad_eta, quad_w, kappa, sigma)
    loss_totale = loss_bc + loss_clp
    loss_totale.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoque {epoch:04d} | "
              f"Loss totale: {loss_totale.item():.2e} | "
              f"CLP: {loss_clp.item():.2e} | "
              f"BC: {loss_bc.item():.2e}")

## 7. Model Training (L-BFGS)
Fine-tuning of the model parameters using L-BFGS.

In [ ]:
def closure():
    optimizer_lbfgs.zero_grad()
    loss_bc = calc_bc_loss(modele, 
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc, quad_mu, quad_eta, quad_w, kappa, sigma)
    loss_totale = loss_bc + loss_clp
    loss_totale.backward()
    return loss_totale

optimizer_lbfgs = optim.LBFGS(modele.parameters(), line_search_fn="strong_wolfe", max_iter=20)
lbfgs_epochs = 500

for epoch in range(lbfgs_epochs):
    loss = optimizer_lbfgs.step(closure)
    if epoch % 20 == 0:
        print(f"Epoque LBFGS {epoch:03d} | Loss totale: {loss.item():.2e}")

## 8. Visualizing the Results and Validation against a 2D DOM Reference

To validate the PINN solution, we compare the predicted incident radiation $G(x, y)$ along the centerlines through the source against a reference computed by a **Discrete Ordinates (DOM) solver** — same approach as for Case 2 — solving exactly the same equation, internal source and (cold) boundary conditions (implicit upwind sweeps + source iteration, grid-converged).

No simple analytical anchor exists for this case (the $\pi$ rotation argument needs $\omega = 1$ and a hot wall). The grid-converged DOM is the reference. Physical sanity check: $G$ must peak at the source centre $(0.5, 0.9)$ and decay smoothly towards the cold walls.

In [ ]:
# Solveur DOM 2D de reference : meme equation, source interne et memes BCs (parois froides).
# Schema upwind implicite, balayages vectorises par fronts d'onde (les cellules
# d'une anti-diagonale ne dependent que de la precedente), iteration de la source.

def resoudre_dom(M=201, N_xi=12, N_phi=48, beta=1.5, sigma_dom=1.0, tol=1e-8, max_iter=2000):
    dx = 1.0 / (M - 1)

    nx, wx = np.polynomial.legendre.leggauss(N_xi)
    xi = 0.5 * (nx + 1.0)
    phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    XI, PHI = np.meshgrid(xi, phi, indexing='ij')
    W = np.outer(wx, np.full(N_phi, 2.0 * np.pi / N_phi)).ravel()
    st = np.sqrt(1.0 - XI**2)
    MU = (st * np.cos(PHI)).ravel()
    ETA = (st * np.sin(PHI)).ravel()

    xs = np.linspace(0.0, 1.0, M)
    Xs, Ys = np.meshgrid(xs, xs, indexing='ij')
    S_src = source_S_np(Xs, Ys)

    quadrants = []
    for smu in (1, -1):
        for seta in (1, -1):
            sel = (np.sign(MU) == smu) & (np.sign(ETA) == seta)
            quadrants.append((smu, seta, MU[sel], ETA[sel], W[sel]))

    def balayage(smu, seta, mu, eta, S):
        a = np.abs(mu)[:, None] / dx
        b = np.abs(eta)[:, None] / dx
        I = np.zeros((mu.size, M, M))
        Sf = S[::smu, ::seta]             # vues retournees : balayage toujours croissant
        If = I[:, ::smu, ::seta]
        denom = a + b + beta
        for d in range(2 * M - 1):
            ii = np.arange(max(0, d - M + 1), min(d, M - 1) + 1)
            jj = d - ii
            Ix = If[:, np.where(ii > 0, ii - 1, 0), jj]
            Ix[:, ii == 0] = 0.0
            Iy = If[:, ii, np.where(jj > 0, jj - 1, 0)]
            Iy[:, jj == 0] = 0.0          # toutes les parois froides
            If[:, ii, jj] = (a * Ix + b * Iy + Sf[ii, jj][None, :]) / denom
        return I

    G = np.zeros((M, M))
    for it in range(max_iter):
        S = (sigma_dom / (4.0 * np.pi)) * G + S_src
        Gn = np.zeros_like(G)
        for smu, seta, mu, eta, w in quadrants:
            Gn += np.tensordot(w, balayage(smu, seta, mu, eta, S), axes=(0, 0))
        diff = np.max(np.abs(Gn - G))
        G = Gn
        if diff < tol:
            break
    print(f"DOM converge en {it} iterations (diff = {diff:.2e})")
    return G

M_dom = 201
G_dom = resoudre_dom(M=M_dom, beta=kappa + sigma, sigma_dom=sigma)
grid_dom = np.linspace(0.0, 1.0, M_dom)
ix_c = int(round(CENTRE[0] * (M_dom - 1)))
iy_c = int(round(CENTRE[1] * (M_dom - 1)))
print(f"G au centre de la source {CENTRE} : {G_dom[ix_c, iy_c]:.4f}")

In [ ]:
def evaluer_G(model, x_grid, y_grid, quad_mu, quad_eta, quad_w, device):
    Nx = len(x_grid)
    Ny = len(y_grid)
    X, Y = np.meshgrid(x_grid, y_grid)

    x_tensor = torch.tensor(X.ravel(), dtype=torch.float32).view(-1, 1).to(device)
    y_tensor = torch.tensor(Y.ravel(), dtype=torch.float32).view(-1, 1).to(device)

    N = x_tensor.shape[0]
    N_q = quad_mu.shape[0]

    x_expanded = x_tensor.repeat(1, N_q)
    y_expanded = y_tensor.repeat(1, N_q)
    mu_expanded = quad_mu.t().repeat(N, 1)
    eta_expanded = quad_eta.t().repeat(N, 1)

    inputs = torch.stack([x_expanded, y_expanded, mu_expanded, eta_expanded], dim=2).view(-1, 4)

    G_list = []
    batch_size = 1000
    with torch.no_grad():
        for i in range(0, N, batch_size):
            start_idx = i * N_q
            end_idx = min((i + batch_size) * N_q, N * N_q)
            inputs_batch = inputs[start_idx:end_idx]
            cur_N = inputs_batch.shape[0] // N_q
            I_preds = model(inputs_batch).view(cur_N, N_q)
            G_batch = torch.sum(I_preds * quad_w.t(), dim=1)
            G_list.append(G_batch)
        G_tensor = torch.cat(G_list)

    return G_tensor.cpu().numpy().reshape(Y.shape)

x_vals = np.linspace(0.0, 1.0, 100)
y_vals = np.linspace(0.0, 1.0, 100)
G_pred = evaluer_G(modele, x_vals, y_vals, quad_mu, quad_eta, quad_w, device)

# Plot Heatmap of G(x, y)
fig, ax = plt.subplots(figsize=(7, 6))
X, Y = np.meshgrid(x_vals, y_vals)
im = ax.pcolormesh(X, Y, G_pred, cmap='jet', shading='auto')
ax.set_title("Incident Radiation $G(x, y)$ (2D PINN, case 3b)")
ax.set_xlabel("Position $x$")
ax.set_ylabel("Position $y$")
fig.colorbar(im, label="Incident Radiation $G$")
plt.tight_layout()
fig.savefig("intensity_heatmap_cas3b.png", dpi=150)
plt.show()

# Plot Line Profiles vs DOM reference (centerlines through the source)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Line x = CENTRE[0]
y_vals_line = np.linspace(0.0, 1.0, 100)
G_xc = evaluer_G(modele, np.array([CENTRE[0]]), y_vals_line, quad_mu, quad_eta, quad_w, device).flatten()
axes[0].plot(y_vals_line, G_xc, color='red', linewidth=2, label="PINN")
axes[0].plot(grid_dom, G_dom[ix_c, :], 'k--', linewidth=1.5, label="DOM (référence)")
axes[0].set_title(f"Incident Radiation $G({CENTRE[0]}, y)$")
axes[0].set_xlabel("Position $y$")
axes[0].set_ylabel(f"$G({CENTRE[0]}, y)$")
axes[0].grid(True)
axes[0].legend()

# Line y = CENTRE[1]
x_vals_line = np.linspace(0.0, 1.0, 100)
G_yc = evaluer_G(modele, x_vals_line, np.array([CENTRE[1]]), quad_mu, quad_eta, quad_w, device).flatten()
axes[1].plot(x_vals_line, G_yc, color='blue', linewidth=2, label="PINN")
axes[1].plot(grid_dom, G_dom[:, iy_c], 'k--', linewidth=1.5, label="DOM (référence)")
axes[1].set_title(f"Incident Radiation $G(x, {CENTRE[1]})$")
axes[1].set_xlabel("Position $x$")
axes[1].set_ylabel(f"$G(x, {CENTRE[1]})$")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
fig.savefig("intensity_boundaries_cas3b.png", dpi=150)
plt.show()

# Erreur relative PINN vs DOM sur la ligne verticale (normalisee par le max)
ref_y = np.linspace(0.0, 1.0, 11)
G_dom_v = np.interp(ref_y, grid_dom, G_dom[ix_c, :])
G_pinn_v = np.interp(ref_y, y_vals_line, G_xc)
err = np.abs(G_pinn_v - G_dom_v) / np.max(np.abs(G_dom_v))
print("y            :", np.round(ref_y, 2))
print("G DOM        :", np.round(G_dom_v, 4))
print("G PINN       :", np.round(G_pinn_v, 4))
print("erreur (% du max) :", np.round(100 * err, 1))

## 9. Angular Distribution of the Intensity: Spheres of Directions

At a fixed spatial point $(x_0, y_0)$, the intensity $I(x_0, y_0, \theta, \phi)$ is a function on the unit sphere of directions. We visualize it as a **radiation pattern**: the sphere is deformed radially by $I$ (radius $\propto I$) and colored by $I$.

Reading guide:
* Axes = directional cosines $(\mu, \eta, \xi)$.
* At the source centre the pattern is nearly isotropic (the internal source emits equally in all directions); away from the source it becomes directional, pointing away from the source.
* The pattern is exactly symmetric under $\xi \to -\xi$: in this 2D configuration $I$ only depends on $(\mu, \eta)$, so the top and bottom hemispheres are mirror images — this is physical, not an artifact.

In [ ]:
def tracer_spheres_intensite(model, points, device, n_theta=60, n_phi=121, deformer=True):
    theta = np.linspace(0.0, np.pi, n_theta)
    phi = np.linspace(0.0, 2.0 * np.pi, n_phi)
    TH, PH = np.meshgrid(theta, phi, indexing='ij')
    MU = np.sin(TH) * np.cos(PH)
    ETA = np.sin(TH) * np.sin(PH)
    XI = np.cos(TH)

    cmap = plt.get_cmap('jet')
    fig = plt.figure(figsize=(5.5 * len(points), 5.5))

    for k, (x0, y0) in enumerate(points):
        inp = np.stack([
            np.full(MU.size, x0),
            np.full(MU.size, y0),
            MU.ravel(),
            ETA.ravel(),
        ], axis=1)
        with torch.no_grad():
            I = model(torch.tensor(inp, dtype=torch.float32).to(device)).cpu().numpy().reshape(MU.shape)
        I = np.clip(I, 0.0, None)

        r = I if deformer else np.ones_like(I)
        norm = plt.Normalize(I.min(), I.max())

        ax = fig.add_subplot(1, len(points), k + 1, projection='3d')
        ax.plot_surface(r * MU, r * ETA, r * XI,
                        facecolors=cmap(norm(I)),
                        rstride=1, cstride=1, linewidth=0,
                        antialiased=False, shade=False)
        L = 1.05 * max(r.max(), 0.1)
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(-L, L)
        ax.set_box_aspect((1, 1, 1))
        ax.set_xlabel(r"$\mu$"); ax.set_ylabel(r"$\eta$"); ax.set_zlabel(r"$\xi$")
        ax.set_title(f"$I({x0}, {y0}, \\theta, \\phi)$")
        m = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        m.set_array(I)
        fig.colorbar(m, ax=ax, shrink=0.6, pad=0.1, label="Intensité $I$")

    plt.tight_layout()
    fig.savefig("intensity_spheres_cas3b.png", dpi=150)
    plt.show()

# au centre de la source, puis deux points a distance croissante
tracer_spheres_intensite(modele, [CENTRE, (0.5, 0.15), (0.85, 0.5)], device)